# 02 - Preprocessing

Based on the findings of `01_eda`: remove duplicates and train/test leakage, split off a validation set,
and save the images in two versions (with and without the black border) for the margin removal experiment.

## 1. Imports

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import imagehash
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

## 2. Paths

In [11]:
DATA_DIR = Path("..") / "data"
PROCESSED_DIR = DATA_DIR / "processed"

## 3. Image table and hashes

Same as in `01_eda`: one row per image with its split, label and perceptual hash.

In [12]:
paths = sorted(DATA_DIR.glob("Training/*/*.jpg")) + sorted(DATA_DIR.glob("Testing/*/*.jpg"))

df = pd.DataFrame({"path": paths})
df["split"] = [p.parent.parent.name for p in paths]
df["label"] = [p.parent.name for p in paths]
df["hash"] = [str(imagehash.phash(Image.open(p))) for p in tqdm(paths)]
df.head()

100%|██████████| 7200/7200 [00:12<00:00, 566.91it/s]


,path,split,label,hash
0,..\data\Training\glioma\Tr-gl_1.jpg,Training,glioma,d01f2b693cb46d70
1,..\data\Training\glioma\Tr-gl_10.jpg,Training,glioma,c09e6b8d2f8b6c2c
2,..\data\Training\glioma\Tr-gl_100.jpg,Training,glioma,c03f2f3d36e43292
3,..\data\Training\glioma\Tr-gl_1000.jpg,Training,glioma,c0967e493e8e2d6c
4,..\data\Training\glioma\Tr-gl_1001.jpg,Training,glioma,93cf6c906166b63c


In [13]:
counts_before = pd.crosstab(df["label"], df["split"], margins=True, margins_name="total")
counts_before

split,Testing,Training,total
label,,,
glioma,400,1400,1800
meningioma,400,1400,1800
notumor,400,1400,1800
pituitary,400,1400,1800
total,1600,5600,7200


## 4. Remove duplicates and leakage

### 4.1 Label conflicts

Images whose hash appears with more than one label; dropped.

In [14]:
labels_per_hash = df.groupby("hash")["label"].nunique()
conflict_hashes = labels_per_hash[labels_per_hash > 1].index

df = df[~df["hash"].isin(conflict_hashes)]
len(df)

7197

### 4.2 Leakage

Training images whose hash also appears in the test set are dropped.

In [15]:
test_hashes = df.loc[df["split"] == "Testing", "hash"]

df = df[~((df["split"] == "Training") & df["hash"].isin(test_hashes))]
len(df)

6713

### 4.3 Duplicates within a split

Keep the first image of every duplicate group, separately for train and test.

In [16]:
df = df.drop_duplicates(subset=["split", "hash"]).reset_index(drop=True)
len(df)

6211

### 4.4 Before and after

In [17]:
counts_after = pd.crosstab(df["label"], df["split"], margins=True, margins_name="total")
pd.concat([counts_before, counts_after], axis=1, keys=["before", "after"])

before                  after               
split      Testing Training total Testing Training total
label                                                   
glioma         400     1400  1800     381     1393  1774
meningioma     400     1400  1800     389     1277  1666
notumor        400     1400  1800     303      720  1023
pituitary      400     1400  1800     400     1348  1748
total         1600     5600  7200    1473     4738  6211

### 4.5 Conclusions

- 7,200 to 6,211 images: 4,738 train and 1,473 test.
- Notumor lost the most: 1,400 to 720 in train and 400 to 303 in test.
- Train and test no longer share any image (by hash), so the test score will be honest.
- The classes are no longer balanced: notumor has about half as many training images. Class weights in `03_training` correct for this.

## 5. Validation split

### 5.1 Split off 15% of train

`stratify` keeps the class proportions the same in train and validation.

In [18]:
df["split"] = df["split"].replace({"Training": "train", "Testing": "test"})

train_rows = df[df["split"] == "train"]
_, val_rows = train_test_split(train_rows, test_size=0.15, stratify=train_rows["label"], random_state=42)
df.loc[val_rows.index, "split"] = "val"

pd.crosstab(df["label"], df["split"], margins=True, margins_name="total")[["train", "val", "test", "total"]]

split,train,val,test,total
label,,,,
glioma,1184,209,381,1774
meningioma,1085,192,389,1666
notumor,612,108,303,1023
pituitary,1146,202,400,1748
total,4027,711,1473,6211


### 5.2 Check the class proportions


In [19]:
pd.crosstab(df["label"], df["split"], normalize="columns").round(3)[["train", "val", "test"]]

split,train,val,test
label,,,
glioma,0.294,0.294,0.259
meningioma,0.269,0.270,0.264
notumor,0.152,0.152,0.206
pituitary,0.285,0.284,0.272


### 5.3 Conclusions

- Final split: 4,027 train, 711 validation and 1,473 test images.
- Train and validation have the same class proportions, thanks to `stratify`.
- The test set has relatively more notumor (20.6%), because fewer test duplicates were removed. That is fine: the test set is only used at the end.